In [1]:
import pandas as pd

df = pd.read_csv("../data/all_island_quarterly_inflation_clean.csv")

df.shape


(2357, 13)

In [2]:
def drop_annual_totals(values):
    # Remove annual totals
    return [v for i, v in enumerate(values) if (i + 1) % 5 != 0]


In [ ]:
belfast_raw = [
599,604,534,606,2343,
585,573,677,714,2549,
744,700,788,797,3029,
729,816,907,1108,3560,
994,1104,1213,1194,4505,
962,1096,1174,1305,4537,
1250,1015,1147,1150,4562,
1058,1139,1286,1220,4703,
1062,1148,1229,1280,4719,
959,1152,1254,1208,4573,
1062,329,960,1566,3917,
1503,1589,1626,1322,6040,
1281,1179,1349,1225,5034,
1074,1013,1248,1104,4439,
922,1113,1264,1260,4559,
1341,905,1159
]

belfast_q = drop_annual_totals(belfast_raw)
len(belfast_q) # print to check there are 63, matching no of quarters


63

In [4]:
ards_raw = [
286,269,286,305,1146,
235,294,357,342,1228,
359,315,411,424,1509,
420,466,491,633,2010,
565,645,702,670,2582,
494,599,693,672,2458,
638,575,716,698,2627,
541,709,680,760,2690,
588,704,760,751,2803,
551,676,843,843,2913,
558,230,622,1052,2462,
976,1023,1001,752,3752,
692,750,804,724,2970,
528,602,693,658,2481,
531,674,755,818,2778,
838,590,738
]

ards_q = drop_annual_totals(ards_raw)
len(ards_q)


63

In [5]:
def build_replacement_df(df, county, values):
    base = (
        df[df["County"] == county]
        .sort_values(["Year", "Quarter"])
        [["Year", "Quarter", "Period"]]
        .reset_index(drop=True)
    )

    assert len(base) == len(values), f"Length mismatch for {county}"

    base["transaction_count"] = values
    base["County"] = county
    return base


In [6]:
belfast_fix = build_replacement_df(df, "Belfast", belfast_q)
ards_fix = build_replacement_df(df, "Ards and North Down", ards_q)


In [8]:
belfast_fix.head(20)

,Year,Quarter,Period,transaction_count,County
0,2010,1,2010 Q1,599,Belfast
1,2010,2,2010 Q2,604,Belfast
2,2010,3,2010 Q3,534,Belfast
3,2010,4,2010 Q4,606,Belfast
4,2011,1,2011 Q1,585,Belfast
5,2011,2,2011 Q2,573,Belfast
6,2011,3,2011 Q3,677,Belfast
7,2011,4,2011 Q4,714,Belfast
8,2012,1,2012 Q1,744,Belfast
9,2012,2,2012 Q2,700,Belfast


In [9]:
df_fixed = df.copy()

for fix_df in [belfast_fix, ards_fix]:
    df_fixed = df_fixed.merge(
        fix_df,
        on=["County", "Year", "Quarter", "Period"],
        how="left",
        suffixes=("", "_new")
    )

    df_fixed["transaction_count"] = (
        df_fixed["transaction_count_new"]
        .combine_first(df_fixed["transaction_count"])
    )

    df_fixed = df_fixed.drop(columns=["transaction_count_new"])


In [11]:
df_fixed.loc[
    df_fixed["County"].isin(["Belfast", "Ards and North Down"]),
    ["County", "Year", "Quarter", "transaction_count", "avg_price"]
].tail(20)


,County,Year,Quarter,transaction_count,avg_price
232,Belfast,2020,4,1566.0,147346.25
233,Belfast,2021,1,1503.0,154284.17
234,Belfast,2021,2,1589.0,160091.83
235,Belfast,2021,3,1626.0,165805.27
236,Belfast,2021,4,1322.0,164632.21
237,Belfast,2022,1,1281.0,172007.30
238,Belfast,2022,2,1179.0,173701.47
239,Belfast,2022,3,1349.0,180468.56
240,Belfast,2022,4,1225.0,174613.46
241,Belfast,2023,1,1074.0,169185.31


In [13]:
df_fixed[
    (df_fixed["County"].isin(["Belfast", "Ards and North Down"])) &
    (df_fixed["transaction_count"] == 0.0)
]


,County,Year,Quarter,Period,Inflation_Rate,Region,is_ni,transaction_count,avg_price,price_qoq_pct,price_yoy_pct,real_price_qoq_pct,real_price_yoy_pct


In [15]:
df_fixed.to_csv(
    "../data/all_island_quarterly_inflation_clean.csv",
    index=False
)
